In [3]:
import numpy as np 
import qutip as qt
import ufss as uf
import matplotlib.pyplot as plt

# 2. La spectroscopie 2D
from qudpy.Classes import System
import qudpy.plot_functions as pf

# Physic constant and definition of parameter

In [4]:
hbar = 0.658211951 # hbar in eV*fs


###### Electronic Energy #######
E_elec= 1.5 # d-d transition energy
E_elec_2= 1.6
E_vib = 0.020 # Phonon énergie

####### Factor and coupling parameter ##########
S=0.01 # Facteur de Hunag-Rhys (For Frank-condon approximation)
lambda_HT= 0.5 # Coupling parameter Herzberg-Teller modele

################### frequency ##################
w_e = E_elec  #electronic transition frequency
w_v = E_vib  #phonon transition frequency
N_fock = 2 # number of phonon mode. 

# Operator definition 

In [5]:
class SpinOrbitalChain:
    def __init__(self, N):
        self.N = N
        self.dim_tot = 4**N
        
        ###### Operator definition #########

        #Pauli Matrix
        Sx = qt.sigmax()/2.0
        Sy = qt.sigmay()/2.0
        Sz = qt.sigmaz()/2.0

        Sp = qt.sigmap() #upper
        Sm = qt.sigmam() #lower

        I2 =  qt.qeye(2) #Identity 2x2

        #Local spin operator (act on the spin, the identity on the orbite)
        self.Sx_loc = qt.tensor(Sx, I2)
        self.Sy_loc = qt.tensor(Sy, I2)
        self.Sz_loc = qt.tensor(Sz, I2)

        # Local orbital operator (identity on spin, act on orbite)

        self.Tx_loc = qt.tensor(I2, Sx)
        self.Ty_loc = qt.tensor(I2, Sy)
        self.Tz_loc = qt.tensor(I2, Sz)

        # orbital ladder operatror

        self.Tp_loc = qt.tensor(I2, Sp)
        self.Tm_loc = qt.tensor(I2, Sm)

    def make_tensor(self, op_loc, site):
        """
        Put a local operator (4X4) on a specific site in a chain of lenght N
        """
        if site < 0 or site >= self.N:
            raise ValueError(f"site need to be between 0 and {self.N-1}")
        self.I_loc = qt.tensor(qt.qeye(2), qt.qeye(2))
        # List of all the local identity on the chain
        op_list = [self.I_loc for _ in range(self.N)]

        op_list[site] = op_loc

        return qt.tensor(*op_list)
    
    def S(self, alpha, site):
        """Retourne l'opérateur de spin S^alpha au site spécifié. alpha = 'x', 'y' ou 'z'"""
        if alpha == 'x': return self.make_tensor(self.Sx_loc, site)
        if alpha == 'y': return self.make_tensor(self.Sy_loc, site)
        if alpha == 'z': return self.make_tensor(self.Sz_loc, site)
        
    def T(self, alpha, site):
        """Retourne l'opérateur orbital T^alpha au site spécifié."""
        if alpha == 'x': return self.make_tensor(self.Tx_loc, site)
        if alpha == 'y': return self.make_tensor(self.Ty_loc, site)
        if alpha == 'z': return self.make_tensor(self.Tz_loc, site)
        if alpha == '+': return self.make_tensor(self.Tp_loc, site)
        if alpha == '-': return self.make_tensor(self.Tm_loc, site)






In [ ]:
if __name__ == "__main__":
    N_sites = 6
    chain = SpinOrbitalChain(N_sites)
    
    # Récupérer l'opérateur S^z au site 0
    Sz_0 = chain.S('z', 0)
    
    # Récupérer l'opérateur T^+ au site 3
    Tp_3 = chain.T('+', 3)
    
    print(f"Dimension de Sz_0 : {Sz_0.shape}") # Devrait afficher (4096, 4096) pour N=6
    
    # Vérification d'une règle de commutation : [Sz_0, Sz_1] = 0
    Sx_1 = chain.S('x', 1)
    commutateur = qt.commutator(Sz_0, Sx_1)
    print(f"Norme du commutateur [Sz_0, Sz_1] : {commutateur.norm()}") # Devrait être 0.0   

    
    commutateur = qt.commutator(Tp_3, Sx_1)
    print(f"Norme du commutateur [Tp_3, Sz_1] : {commutateur.norm()}") # Devrait être 0.0  


Dimension de Sz_0 : (4096, 4096)
Norme du commutateur [Sz_0, Sz_1] : 0.0
Norme du commutateur [Tp_3, Sz_1] : 0.0
